# Figure 5a — Dimensionality reduction of language date-vectors

Each language is represented as a vector of 6-week-summed Twitter frequencies over time.
UMAP or t-SNE reduces those language vectors to 2D, revealing temporal structure.

**Inputs:** `CHOSEN_WEEKLY_PIVOT_FILE` (`data/processed/chosen_words_weekly_pivoted.csv`)
**Outputs:** `outputs/figures/Fig.5a_Vectors_UMAPs/`
**Prerequisites:** run `02_combine_data.ipynb` first; `pip install umap-learn scikit-learn`

In [ ]:
import sys
sys.path.insert(0, '..')
from config import CHOSEN_WEEKLY_PIVOT_FILE, WORD_FORMS_ALL, FIGURES_DIR

import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"

## ⚙️ Parameters

In [ ]:
# Dimensionality-reduction method: 'umap' or 'tsne'
graphtype = 'tsne'

# Distance metric used by the reducer
metric = 'euclidean'

# Tuning variable:
#   graphtype == 'umap'  →  n_neighbors (e.g. 15)
#   graphtype == 'tsne'  →  perplexity  (e.g. 15–30)
variable = 15

## Load data & 6-week aggregation

In [ ]:
DFfreq = pd.read_csv(CHOSEN_WEEKLY_PIVOT_FILE, index_col=0)

# Robust index handling: supports both ISO-index and language-index pivots
meta_lang = pd.read_csv(WORD_FORMS_ALL, usecols=['ISO', 'Language']).drop_duplicates()
iso_to_name = meta_lang.set_index('ISO')['Language'].to_dict()
iso_set = set(meta_lang['ISO'])
name_set = set(meta_lang['Language'])

idx = pd.Index(DFfreq.index.astype(str))
if idx.isin(iso_set).all():
    DFfreq.index = idx.map(iso_to_name)
elif idx.isin(name_set).all():
    DFfreq.index = idx
else:
    DFfreq.index = idx.map(iso_to_name).fillna(idx)
DFfreq.index = pd.Index(DFfreq.index, name='language')

# Keep only date columns, convert to datetime
date_mask = pd.to_datetime(DFfreq.columns, errors='coerce').notna()
DFfreq = DFfreq.loc[:, date_mask].copy()
DFfreq.columns = pd.to_datetime(DFfreq.columns)

# Group into 6-week periods (rows = languages, columns = 6W period end-dates)
df = DFfreq.T.groupby(
    pd.Grouper(freq='6W', label='right', closed='right')
).sum().T

# For the reducer each 6W period is one observation, languages are its features
df_transposed = df.T  # shape: (6W-periods × languages)

print(f"Input shape: {df_transposed.shape}  (6W-periods × languages)")
df_transposed.head(3)

## Dimensionality reduction

In [ ]:
import umap
from sklearn.manifold import TSNE

if graphtype == 'umap':
    reducer = umap.UMAP(
        random_state=42,
        n_neighbors=variable,
        min_dist=0.1,
        metric=metric,
    )
    embedding = reducer.fit_transform(df_transposed.values)
elif graphtype == 'tsne':
    tsne = TSNE(random_state=42, metric=metric, perplexity=variable, n_iter=1000)
    embedding = tsne.fit_transform(df_transposed.values)
else:
    raise ValueError(f"graphtype must be 'umap' or 'tsne', got: {graphtype!r}")

umap_df = pd.DataFrame(embedding, columns=['UMAP1', 'UMAP2'], index=df_transposed.index)
umap_df = umap_df.reset_index().rename(columns={'index': 'date'})
umap_df['date_numeric'] = mdates.date2num(umap_df['date'])

print(f"Embedding shape: {embedding.shape}")
umap_df.head(3)

## Interactive plot (Plotly)

In [ ]:
min_year = umap_df['date'].min().year
max_year = umap_df['date'].max().year
tick_dates  = [pd.Timestamp(year=y, month=1, day=1) for y in range(min_year, max_year + 1)]
tickvals_cb = [mdates.date2num(t) for t in tick_dates]
ticktext_cb = [t.strftime("%Y") for t in tick_dates]

umap_df_sorted = umap_df.sort_values(by='date')
colorscale  = px.colors.sequential.Viridis
n_colors    = len(colorscale)
date_min    = umap_df_sorted['date_numeric'].min()
date_max    = umap_df_sorted['date_numeric'].max()

fig_interactive = go.Figure()

fig_interactive.add_trace(go.Scatter(
    y=umap_df['UMAP1'],
    x=umap_df['UMAP2'],
    mode='markers',
    showlegend=False,
    marker=dict(
        size=7,
        color=umap_df['date_numeric'],
        colorscale='Viridis',
        colorbar=dict(
            title='Date',
            tickmode='array',
            tickvals=tickvals_cb,
            ticktext=ticktext_cb,
        ),
        showscale=True,
    ),
    text=umap_df['date'].astype(str),
    hovertemplate='Date: %{text}<br>Dim1: %{y:.2f}<br>Dim2: %{x:.2f}<extra></extra>',
))

for i in range(1, len(umap_df_sorted)):
    date_val = umap_df_sorted.iloc[i]['date_numeric']
    norm = (date_val - date_min) / (date_max - date_min)
    color = colorscale[int(norm * (n_colors - 1))]
    fig_interactive.add_trace(go.Scatter(
        y=[umap_df_sorted.iloc[i-1]['UMAP1'], umap_df_sorted.iloc[i]['UMAP1']],
        x=[umap_df_sorted.iloc[i-1]['UMAP2'], umap_df_sorted.iloc[i]['UMAP2']],
        mode='lines',
        line=dict(color=color, width=2),
        opacity=0.5,
        hoverinfo='skip',
        showlegend=False,
    ))

fig_interactive.update_layout(
    xaxis_title="Dimension 2",
    yaxis_title="Dimension 1",
    template="plotly_white",
    width=800,
    height=630,
)

fig_interactive.show()

## Publication plot (Matplotlib) — this is saved

In [ ]:
# Tick dates (exclude 2008 as it has only partial data)
tick_dates_mpl  = [pd.Timestamp(year=y, month=1, day=1)
                   for y in range(min_year, max_year + 1) if y != 2008]
tickvals_mpl    = [mdates.date2num(t) for t in tick_dates_mpl]
ticklabels_mpl  = [t.strftime("%Y") for t in tick_dates_mpl]

umap_df_sorted = umap_df.sort_values(by='date_numeric').reset_index(drop=True)
date_min = umap_df_sorted['date_numeric'].min()
date_max = umap_df_sorted['date_numeric'].max()
norm_mpl = mcolors.Normalize(vmin=date_min, vmax=date_max)
cmap_mpl = plt.get_cmap('viridis')

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    umap_df['UMAP2'], umap_df['UMAP1'],
    c=umap_df['date_numeric'], cmap='viridis', s=18,
)

for i in range(1, len(umap_df_sorted)):
    color = cmap_mpl(norm_mpl(umap_df_sorted.loc[i, 'date_numeric']))
    ax.plot(
        [umap_df_sorted.loc[i-1, 'UMAP2'], umap_df_sorted.loc[i, 'UMAP2']],
        [umap_df_sorted.loc[i-1, 'UMAP1'], umap_df_sorted.loc[i, 'UMAP1']],
        color=color, linewidth=2, alpha=0.5,
    )

ax.set_xlabel("Dimension 2", fontsize=11)
ax.xaxis.set_label_position("top")
ax.set_ylabel("Dimension 1", fontsize=11)
ax.tick_params(axis='both', labelsize=9)
ax.invert_yaxis()

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_ticks(tickvals_mpl)
cbar.set_ticklabels(ticklabels_mpl)
cbar.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

## Save figure

In [ ]:
today_str = datetime.now().strftime("%Y%m%d")
fig_dir = FIGURES_DIR / "Fig.5a_Vectors_UMAPs"
basename = f"totalfreq_{graphtype}_{metric}_{variable}_{today_str}"

for fmt in ['pdf', 'svg', 'jpeg']:
    (fig_dir / fmt).mkdir(parents=True, exist_ok=True)

# Publication figure (matplotlib)
fig.savefig(str(fig_dir / "pdf"  / f"{basename}.pdf"),  format='pdf',  bbox_inches='tight')
fig.savefig(str(fig_dir / "jpeg" / f"{basename}.jpg"),  format='jpg',  dpi=500, bbox_inches='tight')
fig.savefig(str(fig_dir / "svg"  / f"{basename}.svg"),  format='svg',  bbox_inches='tight')

# Interactive version (plotly → HTML)
html_path = fig_dir / f"{basename}.html"
fig_interactive.write_html(str(html_path))

print(f"Saved PDF:  {fig_dir / 'pdf'  / f'{basename}.pdf'}")
print(f"Saved JPEG: {fig_dir / 'jpeg' / f'{basename}.jpg'}")
print(f"Saved SVG:  {fig_dir / 'svg'  / f'{basename}.svg'}")
print(f"Saved HTML: {html_path}")